In [0]:
SELECT
    COUNT(*) AS linhas,
    COUNT(DISTINCT `Row ID`) AS row_ids_distintos,
    COUNT(*) - COUNT(DISTINCT `Row ID`) AS row_ids_repetidos,
    COUNT(DISTINCT `Order ID`) AS pedidos_distintos,
    MIN(`Order Date`) AS primeira_compra,
    MAX(`Order Date`) AS ultima_compra,

    SUM(CASE WHEN `Row ID` IS NULL THEN 1 ELSE 0 END) AS row_id_nulo,
    SUM(CASE WHEN `Order ID` IS NULL OR TRIM(`Order ID`) = ''
             THEN 1 ELSE 0 END) AS pedido_ausente,
    SUM(CASE WHEN `Order Date` IS NULL THEN 1 ELSE 0 END) AS data_compra_nula,
    SUM(CASE WHEN `Ship Date` < `Order Date`
             THEN 1 ELSE 0 END) AS envio_antes_da_compra,
    SUM(CASE WHEN Sales IS NULL OR Sales <= 0
             THEN 1 ELSE 0 END) AS vendas_invalidas,
    SUM(CASE WHEN Quantity IS NULL OR Quantity <= 0
             THEN 1 ELSE 0 END) AS quantidades_invalidas,
    SUM(CASE WHEN Discount IS NULL OR Discount < 0 OR Discount > 1
             THEN 1 ELSE 0 END) AS descontos_invalidos,
    SUM(CASE WHEN Profit IS NULL THEN 1 ELSE 0 END) AS lucros_nulos,
    SUM(CASE WHEN `Postal Code` IS NULL THEN 1 ELSE 0 END) AS ceps_nulos,

    ROUND(SUM(Sales), 2) AS vendas_controle,
    ROUND(SUM(Profit), 2) AS lucro_controle
FROM workspace.default.sample_superstore;

linhas,row_ids_distintos,row_ids_repetidos,pedidos_distintos,primeira_compra,ultima_compra,row_id_nulo,pedido_ausente,data_compra_nula,envio_antes_da_compra,vendas_invalidas,quantidades_invalidas,descontos_invalidos,lucros_nulos,ceps_nulos,vendas_controle,lucro_controle
9994,9994,0,5009,2014-01-03,2017-12-30,0,0,0,0,0,0,0,0,0,2297200.86,286397.02


In [0]:
WITH pedidos AS (
    SELECT
        `Order ID` AS order_id,
        COUNT(DISTINCT `Customer ID`) AS clientes,
        COUNT(DISTINCT `Order Date`) AS datas_compra,
        COUNT(DISTINCT `Ship Date`) AS datas_envio
    FROM workspace.default.sample_superstore
    GROUP BY `Order ID`
),
produtos AS (
    SELECT
        `Product ID` AS product_id,
        COUNT(DISTINCT Category) AS categorias,
        COUNT(DISTINCT `Sub-Category`) AS subcategorias,
        COUNT(DISTINCT `Product Name`) AS nomes
    FROM workspace.default.sample_superstore
    GROUP BY `Product ID`
)
SELECT
    (SELECT COUNT(*) FROM pedidos WHERE clientes > 1)
        AS pedidos_com_clientes_diferentes,
    (SELECT COUNT(*) FROM pedidos WHERE datas_compra > 1)
        AS pedidos_com_datas_compra_diferentes,
    (SELECT COUNT(*) FROM pedidos WHERE datas_envio > 1)
        AS pedidos_com_datas_envio_diferentes,
    (SELECT COUNT(*) FROM produtos WHERE categorias > 1)
        AS produtos_em_categorias_diferentes,
    (SELECT COUNT(*) FROM produtos WHERE subcategorias > 1)
        AS produtos_em_subcategorias_diferentes,
    (SELECT COUNT(*) FROM produtos WHERE nomes > 1)
        AS produtos_com_nomes_diferentes;

pedidos_com_clientes_diferentes,pedidos_com_datas_compra_diferentes,pedidos_com_datas_envio_diferentes,produtos_em_categorias_diferentes,produtos_em_subcategorias_diferentes,produtos_com_nomes_diferentes
0,0,0,0,0,32


In [0]:
SELECT
    `Product ID` AS product_id,
    Category AS categoria,
    `Sub-Category` AS subcategoria,
    COUNT(DISTINCT `Product Name`) AS nomes_distintos,
    COUNT(*) AS linhas,
    CONCAT_WS(' | ', SORT_ARRAY(COLLECT_SET(`Product Name`))) AS nomes_encontrados
FROM workspace.default.sample_superstore
GROUP BY `Product ID`, Category, `Sub-Category`
HAVING COUNT(DISTINCT `Product Name`) > 1
ORDER BY nomes_distintos DESC, linhas DESC, product_id;

product_id,categoria,subcategoria,nomes_distintos,linhas,nomes_encontrados
OFF-PA-10001970,Office Supplies,Paper,2,19,Xerox 1881 | Xerox 1908
TEC-AC-10003832,Technology,Accessories,2,18,Imation�16GB Mini TravelDrive USB 2.0�Flash Drive | Logitech�P710e Mobile Speakerphone
FUR-FU-10004270,Furniture,Furnishings,2,16,"Eldon Image Series Desk Accessories, Burgundy | Executive Impressions 13"" Clairmont Wall Clock"
FUR-CH-10001146,Furniture,Chairs,2,15,"Global Task Chair, Black | Global Value Mid-Back Manager's Chair, Gray"
TEC-AC-10002049,Technology,Accessories,2,15,Logitech G19 Programmable Gaming Keyboard | Plantronics Savi W720 Multi-Device Wireless Headset System
FUR-FU-10001473,Furniture,Furnishings,2,14,"DAX Wood Document Frame | Eldon Executive Woodline II Desk Accessories, Mahogany"
OFF-BI-10002026,Office Supplies,Binders,2,14,Avery Arch Ring Binders | Ibico Recycled Linen-Style Covers
OFF-BI-10004632,Office Supplies,Binders,2,14,GBC Binding covers | Ibico Hi-Tech Manual Binding System
OFF-PA-10002377,Office Supplies,Paper,2,14,"Adams Telephone Message Book W/Dividers/Space For Phone Numbers, 5 1/4""X8 1/2"", 200/Messages | Xerox 1916"
FUR-FU-10004864,Furniture,Furnishings,2,13,"Eldon 500 Class Desk Accessories | Howard Miller 14-1/2"" Diameter Chrome Round Wall Clock"


In [0]:
WITH ids_ambiguos AS (
    SELECT `Product ID`
    FROM workspace.default.sample_superstore
    GROUP BY `Product ID`
    HAVING COUNT(DISTINCT `Product Name`) > 1
),
pares AS (
    SELECT
        `Product ID`,
        `Product Name`,
        COUNT(DISTINCT Category) AS categorias,
        COUNT(DISTINCT `Sub-Category`) AS subcategorias
    FROM workspace.default.sample_superstore
    GROUP BY `Product ID`, `Product Name`
)
SELECT
    COUNT(*) AS linhas_afetadas,
    COUNT(DISTINCT s.`Product ID`) AS ids_ambiguos,
    ROUND(SUM(s.Sales), 2) AS vendas_afetadas,
    ROUND(SUM(s.Profit), 2) AS lucro_afetado,
    SUM(CASE WHEN INSTR(s.`Product Name`, '�') > 0
             THEN 1 ELSE 0 END) AS linhas_com_caractere_substituido,
    (SELECT COUNT(*)
     FROM pares
     WHERE categorias > 1 OR subcategorias > 1) AS pares_com_categoria_inconsistente
FROM workspace.default.sample_superstore s
INNER JOIN ids_ambiguos a
    ON s.`Product ID` = a.`Product ID`;

linhas_afetadas,ids_ambiguos,vendas_afetadas,lucro_afetado,linhas_com_caractere_substituido,pares_com_categoria_inconsistente
337,32,97465.67,13230.92,27,0


In [0]:
CREATE OR REPLACE TABLE workspace.default.superstore_silver_order_lines
USING DELTA
AS
SELECT
    CAST(`Row ID` AS BIGINT) AS order_line_id,
    TRIM(`Order ID`) AS order_id,
    CAST(`Order Date` AS DATE) AS order_date,
    CAST(`Ship Date` AS DATE) AS ship_date,
    TRIM(`Ship Mode`) AS ship_mode,

    TRIM(`Customer ID`) AS customer_id,
    TRIM(`Customer Name`) AS customer_name,
    TRIM(Segment) AS segment,
    TRIM(Country) AS country,
    TRIM(City) AS city,
    TRIM(State) AS state,
    CAST(`Postal Code` AS STRING) AS postal_code,
    TRIM(Region) AS region,

    TRIM(`Product ID`) AS source_product_id,
    TRIM(`Product Name`) AS product_name,
    SHA2(
        TO_JSON(NAMED_STRUCT(
            'product_id', TRIM(`Product ID`),
            'product_name', TRIM(`Product Name`)
        )),
        256
    ) AS product_key,
    TRIM(Category) AS category,
    TRIM(`Sub-Category`) AS subcategory,

    CAST(Sales AS DECIMAL(18, 4)) AS sales,
    CAST(Quantity AS INT) AS quantity,
    CAST(Discount AS DECIMAL(6, 4)) AS discount,
    CAST(Profit AS DECIMAL(18, 4)) AS profit,

    INSTR(`Product Name`, '�') > 0 AS has_replacement_character
FROM workspace.default.sample_superstore;

num_affected_rows,num_inserted_rows


In [0]:
SELECT
    COUNT(*) AS linhas,
    COUNT(DISTINCT order_line_id) AS linhas_distintas,
    COUNT(DISTINCT product_key) AS produtos_por_chave_composta,
    ROUND(SUM(sales), 2) AS vendas,
    ROUND(SUM(profit), 2) AS lucro,
    SUM(CASE WHEN has_replacement_character THEN 1 ELSE 0 END)
        AS linhas_com_caractere_substituido
FROM workspace.default.superstore_silver_order_lines;

linhas,linhas_distintas,produtos_por_chave_composta,vendas,lucro,linhas_com_caractere_substituido
9994,9994,1894,2297200.86,286397.02,278


In [0]:
WITH origem AS (
    SELECT
        COUNT(*) AS linhas,
        ROUND(SUM(Sales), 2) AS vendas,
        ROUND(SUM(Profit), 2) AS lucro
    FROM workspace.default.sample_superstore
),
tratada AS (
    SELECT
        COUNT(*) AS linhas,
        COUNT(DISTINCT order_line_id) AS ids_linha_distintos,
        SUM(CASE WHEN product_key IS NULL THEN 1 ELSE 0 END)
            AS chaves_produto_nulas,
        SUM(CASE WHEN ship_date < order_date THEN 1 ELSE 0 END)
            AS envios_antes_da_compra,
        SUM(CASE WHEN sales <= 0 OR quantity <= 0
                       OR discount < 0 OR discount > 1
                 THEN 1 ELSE 0 END) AS linhas_invalidas,
        ROUND(SUM(sales), 2) AS vendas,
        ROUND(SUM(profit), 2) AS lucro
    FROM workspace.default.superstore_silver_order_lines
)
SELECT
    CASE WHEN t.linhas = o.linhas
         THEN 'PASSOU' ELSE 'FALHOU' END AS contagem_linhas,
    CASE WHEN t.ids_linha_distintos = t.linhas
         THEN 'PASSOU' ELSE 'FALHOU' END AS unicidade_linha,
    CASE WHEN t.chaves_produto_nulas = 0
         THEN 'PASSOU' ELSE 'FALHOU' END AS chave_produto,
    CASE WHEN t.envios_antes_da_compra = 0
              AND t.linhas_invalidas = 0
         THEN 'PASSOU' ELSE 'FALHOU' END AS regras_basicas,
    CASE WHEN t.vendas = o.vendas AND t.lucro = o.lucro
         THEN 'PASSOU' ELSE 'FALHOU' END AS conciliacao_financeira
FROM origem o
CROSS JOIN tratada t;

contagem_linhas,unicidade_linha,chave_produto,regras_basicas,conciliacao_financeira
PASSOU,PASSOU,PASSOU,PASSOU,PASSOU


In [0]:
CREATE OR REPLACE TABLE workspace.default.superstore_gold_month_region_category
USING DELTA
AS
SELECT
    DATE_TRUNC('MONTH', order_date) AS sales_month,
    region,
    category,
    COUNT(*) AS order_lines,
    SUM(quantity) AS units,
    ROUND(SUM(sales), 4) AS sales,
    ROUND(SUM(profit), 4) AS profit
FROM workspace.default.superstore_silver_order_lines
GROUP BY
    DATE_TRUNC('MONTH', order_date),
    region,
    category;

num_affected_rows,num_inserted_rows


In [0]:
SELECT
    COUNT(*) AS grupos,
    COUNT(DISTINCT sales_month) AS meses,
    SUM(order_lines) AS linhas_representadas,
    ROUND(SUM(sales), 2) AS vendas,
    ROUND(SUM(profit), 2) AS lucro,
    MIN(sales_month) AS primeiro_mes,
    MAX(sales_month) AS ultimo_mes
FROM workspace.default.superstore_gold_month_region_category;

grupos,meses,linhas_representadas,vendas,lucro,primeiro_mes,ultimo_mes
573,48,9994,2297200.86,286397.02,2014-01-01T00:00:00.000Z,2017-12-01T00:00:00.000Z


In [0]:
SELECT
    YEAR(sales_month) AS ano,
    COUNT(DISTINCT sales_month) AS meses,
    ROUND(SUM(sales), 2) AS vendas,
    ROUND(SUM(profit), 2) AS lucro,
    ROUND(100 * SUM(profit) / NULLIF(SUM(sales), 0), 2)
        AS margem_pct,
    SUM(order_lines) AS linhas
FROM workspace.default.superstore_gold_month_region_category
GROUP BY YEAR(sales_month)
ORDER BY ano;

ano,meses,vendas,lucro,margem_pct,linhas
2014,12,484247.50,49543.97,10.23,1993
2015,12,470532.51,61618.60,13.10,2102
2016,12,609205.60,81795.17,13.43,2587
2017,12,733215.26,93439.27,12.74,3312


In [0]:
SELECT
    YEAR(sales_month) AS ano,
    category AS categoria,
    ROUND(SUM(sales), 2) AS vendas,
    ROUND(SUM(profit), 2) AS lucro,
    ROUND(100 * SUM(profit) / NULLIF(SUM(sales), 0), 2)
        AS margem_pct
FROM workspace.default.superstore_gold_month_region_category
GROUP BY YEAR(sales_month), category
ORDER BY ano, vendas DESC;

ano,categoria,vendas,lucro,margem_pct
2014,Technology,175278.23,21492.83,12.26
2014,Furniture,157192.85,5457.73,3.47
2014,Office Supplies,151776.41,22593.42,14.89
2015,Furniture,170518.24,3015.20,1.77
2015,Technology,162780.81,33503.87,20.58
2015,Office Supplies,137233.46,25099.53,18.29
2016,Technology,226364.18,39773.99,17.57
2016,Furniture,198901.44,6959.95,3.50
2016,Office Supplies,183939.98,35061.23,19.06
2017,Technology,271730.81,50684.26,18.65


In [0]:
WITH classificada AS (
    SELECT
        YEAR(order_date) AS ano,
        category,
        CASE
            WHEN discount = 0    THEN '00 | Sem desconto'
            WHEN discount <= 0.10 THEN '01 | Até 10%'
            WHEN discount <= 0.20 THEN '02 | Mais de 10% até 20%'
            WHEN discount <= 0.30 THEN '03 | Mais de 20% até 30%'
            ELSE '04 | Mais de 30%'
        END AS faixa_desconto,
        sales,
        profit
    FROM workspace.default.superstore_silver_order_lines
)
SELECT
    ano,
    category,
    faixa_desconto,
    COUNT(*) AS linhas,
    ROUND(SUM(sales), 2) AS vendas,
    ROUND(SUM(profit), 2) AS lucro,
    ROUND(100 * SUM(profit) / NULLIF(SUM(sales), 0), 2)
        AS margem_pct
FROM classificada
GROUP BY ano, category, faixa_desconto
ORDER BY ano, category, faixa_desconto;

ano,category,faixa_desconto,linhas,vendas,lucro,margem_pct
2014,Furniture,00 | Sem desconto,156,52452.47,11590.83,22.10
2014,Furniture,01 | Até 10%,12,8241.09,1141.90,13.86
2014,Furniture,02 | Mais de 10% até 20%,138,54352.92,2099.24,3.86
2014,Furniture,03 | Mais de 20% até 30%,49,26121.96,-2137.54,-8.18
2014,Furniture,04 | Mais de 30%,66,16024.40,-7236.70,-45.16
2014,Office Supplies,00 | Sem desconto,625,99667.37,26701.88,26.79
2014,Office Supplies,01 | Até 10%,3,480.77,143.19,29.78
2014,Office Supplies,02 | Mais de 10% até 20%,458,42884.69,7286.08,16.99
2014,Office Supplies,04 | Mais de 30%,131,8743.58,-11537.73,-131.96
2014,Technology,00 | Sem desconto,156,62839.89,20324.46,32.34


In [0]:
SELECT
    order_line_id,
    order_date,
    order_id,
    source_product_id,
    product_name,
    subcategory,
    quantity,
    discount,
    sales,
    profit,
    ROUND(100 * profit / NULLIF(sales, 0), 2) AS margem_pct
FROM workspace.default.superstore_silver_order_lines
WHERE YEAR(order_date) = 2017
  AND category = 'Office Supplies'
  AND discount > 0.30
ORDER BY profit ASC
LIMIT 20;

order_line_id,order_date,order_id,source_product_id,product_name,subcategory,quantity,discount,sales,profit,margem_pct
4992,2017-12-07,US-2017-122714,OFF-BI-10001120,Ibico EPK-21 Electric Binding System,Binders,5,0.8000,1889.9900,-2929.4845,-155.00
5311,2017-11-19,CA-2017-131254,OFF-BI-10003527,Fellowes PB500 Electric Punch Plastic Comb Binding Machine with Manual Bind,Binders,6,0.8000,1525.1880,-2287.7820,-150.00
2929,2017-10-19,US-2017-120390,OFF-BI-10004995,GBC DocuBind P400 Electric Binding System,Binders,4,0.7000,1633.1880,-1306.5504,-80.00
7899,2017-08-13,CA-2017-128363,OFF-BI-10001359,GBC DocuBind TL300 Electric Binding System,Binders,6,0.7000,1614.5820,-1237.8462,-76.67
2847,2017-09-10,CA-2017-152093,OFF-BI-10003527,Fellowes PB500 Electric Punch Plastic Comb Binding Machine with Manual Bind,Binders,3,0.8000,762.5940,-1143.8910,-150.00
8641,2017-01-12,US-2017-148551,OFF-BI-10000545,GBC Ibimaster 500 Manual ProClick Binding System,Binders,5,0.8000,760.9800,-1141.4700,-150.00
1996,2017-12-02,US-2017-147221,OFF-AP-10002534,3.6 Cubic Foot Counter Height Office Refrigerator,Appliances,5,0.8000,294.6200,-766.0120,-260.00
4498,2017-12-02,CA-2017-155075,OFF-BI-10003650,GBC DocuBind 300 Electric Binding Machine,Binders,4,0.7000,631.1760,-462.8624,-73.33
6567,2017-02-05,CA-2017-131282,OFF-BI-10004632,Ibico Hi-Tech Manual Binding System,Binders,4,0.8000,243.9920,-426.9860,-175.00
3085,2017-02-09,CA-2017-118773,OFF-BI-10004584,GBC ProClick 150 Presentation Binding System,Binders,4,0.8000,252.7840,-417.0936,-165.00


In [0]:
WITH perdas AS (
    SELECT
        subcategory,
        profit,
        ROW_NUMBER() OVER (ORDER BY profit ASC, order_line_id)
            AS posicao
    FROM workspace.default.superstore_silver_order_lines
    WHERE YEAR(order_date) = 2017
      AND category = 'Office Supplies'
      AND discount > 0.30
      AND profit < 0
)
SELECT
    COUNT(*) AS linhas_com_prejuizo,
    ROUND(SUM(profit), 2) AS prejuizo_bruto,
    ROUND(SUM(CASE WHEN posicao <= 5 THEN profit ELSE 0 END), 2)
        AS prejuizo_5_piores,
    ROUND(SUM(CASE WHEN posicao <= 20 THEN profit ELSE 0 END), 2)
        AS prejuizo_20_piores,
    ROUND(SUM(CASE WHEN subcategory = 'Binders' THEN profit ELSE 0 END), 2)
        AS prejuizo_binders,
    ROUND(SUM(CASE WHEN subcategory = 'Appliances' THEN profit ELSE 0 END), 2)
        AS prejuizo_appliances
FROM perdas;

linhas_com_prejuizo,prejuizo_bruto,prejuizo_5_piores,prejuizo_20_piores,prejuizo_binders,prejuizo_appliances
232,-18437.59,-8905.55,-14592.77,-15800.57,-2637.02


In [0]:
DESCRIBE EXTENDED workspace.default.sample_superstore;

col_name,data_type,comment
Row ID,bigint,null
Order ID,string,null
Order Date,date,null
Ship Date,date,null
Ship Mode,string,null
Customer ID,string,null
Customer Name,string,null
Segment,string,null
Country,string,null
City,string,null


In [0]:
SELECT
    LENGTH(CAST(`Postal Code` AS STRING)) AS digitos_armazenados,
    COUNT(*) AS linhas,
    COUNT(DISTINCT CAST(`Postal Code` AS STRING)) AS ceps_distintos,
    CONCAT_WS(', ', SORT_ARRAY(COLLECT_SET(State))) AS estados
FROM workspace.default.sample_superstore
GROUP BY LENGTH(CAST(`Postal Code` AS STRING))
ORDER BY digitos_armazenados;

digitos_armazenados,linhas,ceps_distintos,estados
4,449,52,"Connecticut, Maine, Massachusetts, New Hampshire, New Jersey, Rhode Island, Vermont"
5,9545,579,"Alabama, Arizona, Arkansas, California, Colorado, Delaware, District of Columbia, Florida, Georgia, Idaho, Illinois, Indiana, Iowa, Kansas, Kentucky, Louisiana, Maryland, Michigan, Minnesota, Mississippi, Missouri, Montana, Nebraska, Nevada, New Mexico, New York, North Carolina, North Dakota, Ohio, Oklahoma, Oregon, Pennsylvania, South Carolina, South Dakota, Tennessee, Texas, Utah, Virginia, Washington, West Virginia, Wisconsin, Wyoming"


In [0]:
CREATE OR REPLACE TABLE workspace.default.superstore_silver_order_lines
USING DELTA
AS
SELECT
    CAST(`Row ID` AS BIGINT) AS order_line_id,
    TRIM(`Order ID`) AS order_id,
    CAST(`Order Date` AS DATE) AS order_date,
    CAST(`Ship Date` AS DATE) AS ship_date,
    TRIM(`Ship Mode`) AS ship_mode,

    TRIM(`Customer ID`) AS customer_id,
    TRIM(`Customer Name`) AS customer_name,
    TRIM(Segment) AS segment,
    TRIM(Country) AS country,
    TRIM(City) AS city,
    TRIM(State) AS state,
    CAST(`Postal Code` AS STRING) AS postal_code_source,
    CASE
        WHEN TRIM(Country) = 'United States'
             AND LENGTH(CAST(`Postal Code` AS STRING)) = 4
        THEN LPAD(CAST(`Postal Code` AS STRING), 5, '0')
        ELSE CAST(`Postal Code` AS STRING)
    END AS postal_code,
    (
        TRIM(Country) = 'United States'
        AND LENGTH(CAST(`Postal Code` AS STRING)) = 4
    ) AS postal_code_inferred_zero,
    TRIM(Region) AS region,

    TRIM(`Product ID`) AS source_product_id,
    TRIM(`Product Name`) AS product_name,
    SHA2(
        TO_JSON(NAMED_STRUCT(
            'product_id', TRIM(`Product ID`),
            'product_name', TRIM(`Product Name`)
        )),
        256
    ) AS product_key,
    TRIM(Category) AS category,
    TRIM(`Sub-Category`) AS subcategory,

    CAST(Sales AS DECIMAL(18, 4)) AS sales,
    CAST(Quantity AS INT) AS quantity,
    CAST(Discount AS DECIMAL(6, 4)) AS discount,
    CAST(Profit AS DECIMAL(18, 4)) AS profit,
    INSTR(`Product Name`, '�') > 0 AS has_replacement_character
FROM workspace.default.sample_superstore;

num_affected_rows,num_inserted_rows


In [0]:
SELECT
    COUNT(*) AS linhas,
    SUM(CASE WHEN postal_code_inferred_zero THEN 1 ELSE 0 END)
        AS ceps_com_zero_inferido,
    SUM(CASE WHEN postal_code_inferred_zero
                  AND LEFT(postal_code, 1) = '0'
                  AND LENGTH(postal_code) = 5
             THEN 1 ELSE 0 END) AS ceps_formatados,
    ROUND(SUM(sales), 2) AS vendas,
    ROUND(SUM(profit), 2) AS lucro
FROM workspace.default.superstore_silver_order_lines;

linhas,ceps_com_zero_inferido,ceps_formatados,vendas,lucro
9994,449,449,2297200.86,286397.02


In [0]:
WITH meses AS (
    SELECT EXPLODE(
        SEQUENCE(DATE '2014-01-01', DATE '2017-12-01', INTERVAL 1 MONTH)
    ) AS sales_month
),
regioes AS (
    SELECT DISTINCT region
    FROM workspace.default.superstore_silver_order_lines
),
categorias AS (
    SELECT DISTINCT category
    FROM workspace.default.superstore_silver_order_lines
)
SELECT
    m.sales_month,
    r.region,
    c.category
FROM meses m
CROSS JOIN regioes r
CROSS JOIN categorias c
LEFT JOIN workspace.default.superstore_gold_month_region_category g
    ON g.sales_month = m.sales_month
   AND g.region = r.region
   AND g.category = c.category
WHERE g.sales_month IS NULL
ORDER BY m.sales_month, r.region, c.category;

sales_month,region,category
2014-01-01,West,Technology
2014-02-01,East,Furniture
2015-01-01,South,Technology


In [0]:
CREATE OR REPLACE TABLE workspace.default.superstore_features_month_region_category
USING DELTA AS
WITH months AS (
  SELECT EXPLODE(SEQUENCE(DATE '2014-01-01', DATE '2017-12-01',
                          INTERVAL 1 MONTH)) AS sales_month
), regions AS (
  SELECT DISTINCT region FROM workspace.default.superstore_silver_order_lines
), categories AS (
  SELECT DISTINCT category FROM workspace.default.superstore_silver_order_lines
), dense AS (
  SELECT m.sales_month, r.region, c.category,
         CASE WHEN g.sales_month IS NULL THEN FALSE ELSE TRUE END AS observed_in_source,
         COALESCE(g.sales, CAST(0 AS DECIMAL(18,4))) AS sales,
         COALESCE(g.profit, CAST(0 AS DECIMAL(18,4))) AS profit,
         COALESCE(g.order_lines, 0) AS order_lines
  FROM months m CROSS JOIN regions r CROSS JOIN categories c
  LEFT JOIN workspace.default.superstore_gold_month_region_category g
    ON CAST(g.sales_month AS DATE) = m.sales_month
   AND g.region = r.region
   AND g.category = c.category
)
SELECT sales_month, region, category, observed_in_source,
       sales, profit, order_lines,
       MONTH(sales_month) AS calendar_month,
       LAG(sales, 1) OVER w AS sales_lag_1,
       LAG(sales, 12) OVER w AS sales_lag_12,
       AVG(sales) OVER (
         PARTITION BY region, category ORDER BY sales_month
         ROWS BETWEEN 3 PRECEDING AND 1 PRECEDING
       ) AS sales_trailing_3
FROM dense
WINDOW w AS (PARTITION BY region, category ORDER BY sales_month);

SELECT COUNT(*) AS rows_count,
       SUM(CASE WHEN NOT observed_in_source THEN 1 ELSE 0 END) AS zero_filled_groups,
       SUM(CASE WHEN sales_lag_12 IS NOT NULL THEN 1 ELSE 0 END) AS seasonal_lag_rows,
       ROUND(SUM(sales),2) AS sales,
       ROUND(SUM(profit),2) AS profit
FROM workspace.default.superstore_features_month_region_category;

rows_count,zero_filled_groups,seasonal_lag_rows,sales,profit
576,3,432,2297200.86,286397.02


In [0]:
WITH source AS (
  SELECT sales_month, region, category, CAST(sales AS DOUBLE) AS actual,
         CAST(sales_lag_1 AS DOUBLE) AS lag_1,
         CAST(sales_lag_12 AS DOUBLE) AS lag_12,
         CAST(sales_trailing_3 AS DOUBLE) AS trailing_3
  FROM workspace.default.superstore_features_month_region_category
  WHERE YEAR(sales_month) = 2016
), predictions AS (
  SELECT 'ultimo_mes' AS baseline, category, actual, lag_1 AS predicted FROM source
  UNION ALL
  SELECT 'mes_ano_anterior', category, actual, lag_12 FROM source
  UNION ALL
  SELECT 'media_3_meses', category, actual, trailing_3 FROM source
)
SELECT baseline, category, COUNT(*) AS observations,
       ROUND(AVG(ABS(actual - predicted)),2) AS mae_usd,
       ROUND(100 * SUM(ABS(actual - predicted))
             / NULLIF(SUM(actual),0),2) AS wape_pct,
       ROUND(SUM(predicted - actual),2) AS signed_bias_usd
FROM predictions
WHERE predicted IS NOT NULL
GROUP BY baseline, category
ORDER BY category, wape_pct;

baseline,category,observations,mae_usd,wape_pct,signed_bias_usd
mes_ano_anterior,Furniture,48,2216.54,53.49,-28383.2
media_3_meses,Furniture,48,2467.6,59.55,-14143.41
ultimo_mes,Furniture,48,2673.49,64.52,-13592.9
media_3_meses,Office Supplies,48,2114.7,55.18,-23844.79
mes_ano_anterior,Office Supplies,48,2264.34,59.09,-46706.52
ultimo_mes,Office Supplies,48,2332.01,60.86,-21795.89
mes_ano_anterior,Technology,48,3149.22,66.78,-63583.37
media_3_meses,Technology,48,3854.55,81.73,4188.13
ultimo_mes,Technology,48,4447.92,94.32,13309.27


In [0]:
%python
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.regression import LinearRegression, RandomForestRegressor, GBTRegressor
from pyspark.sql import functions as F

SOURCE = "workspace.default.superstore_features_month_region_category"
raw = spark.table(SOURCE)
df = raw.select(
    F.col("sales_month"),
    F.col("region"),
    F.col("category"),
    F.col("calendar_month"),
    F.col("sales").cast("double").alias("label"),
    F.col("sales_lag_1").cast("double").alias("sales_lag_1"),
    F.col("sales_lag_12").cast("double").alias("sales_lag_12"),
    F.col("sales_trailing_3").cast("double").alias("sales_trailing_3"),
).where(
    F.col("sales_lag_1").isNotNull()
    & F.col("sales_lag_12").isNotNull()
    & F.col("sales_trailing_3").isNotNull()
)

train = df.where(F.year("sales_month") == 2015)
valid = df.where(F.year("sales_month") == 2016)
assert train.count() == 144 and valid.count() == 144, "Unexpected train/validation size"

indexers = [
    StringIndexer(inputCol=c, outputCol=c + "_idx", handleInvalid="keep")
    for c in ["region", "category", "calendar_month"]
]
encoder = OneHotEncoder(
    inputCols=[c + "_idx" for c in ["region", "category", "calendar_month"]],
    outputCols=[c + "_vec" for c in ["region", "category", "calendar_month"]],
    handleInvalid="keep",
)
assembler = VectorAssembler(
    inputCols=["region_vec", "category_vec", "calendar_month_vec",
               "sales_lag_1", "sales_lag_12", "sales_trailing_3"],
    outputCol="features",
)
models = [
    ("ridge", LinearRegression(featuresCol="features", labelCol="label",
                               regParam=1.0, elasticNetParam=0.0, maxIter=100)),
    ("random_forest", RandomForestRegressor(featuresCol="features", labelCol="label",
                                             numTrees=60, maxDepth=4, seed=42)),
    ("gradient_boosted_trees", GBTRegressor(featuresCol="features", labelCol="label",
                                             maxIter=40, maxDepth=3, seed=42)),
]

rows = []
for name, estimator in models:
    fitted = Pipeline(stages=indexers + [encoder, assembler, estimator]).fit(train)
    scored = fitted.transform(valid).select(
        "category", "label",
        # Sales cannot be negative. Fixed postprocessing for all models.
        F.greatest(F.lit(0.0), F.col("prediction")).alias("predicted")
    )
    metrics = scored.groupBy("category").agg(
        F.count("*").alias("n"),
        F.avg(F.abs(F.col("label") - F.col("predicted"))).alias("mae"),
        F.sum(F.abs(F.col("label") - F.col("predicted"))).alias("abs_error"),
        F.sum("label").alias("actual_total"),
        F.sum(F.col("predicted") - F.col("label")).alias("bias")
    ).collect()
    for m in metrics:
        rows.append((
            name, m["category"], m["n"],
            round(m["mae"], 2),
            round(100 * m["abs_error"] / m["actual_total"], 2),
            round(m["bias"], 2)
        ))

result = spark.createDataFrame(
    rows, "model string, category string, observations long, "
          "mae_usd double, wape_pct double, signed_bias_usd double"
)
display(result.orderBy("category", "wape_pct"))

model,category,observations,mae_usd,wape_pct,signed_bias_usd
ridge,Furniture,48,1689.85,40.78,-25592.8
random_forest,Furniture,48,1955.3,47.19,-33805.11
gradient_boosted_trees,Furniture,48,2122.9,51.23,-32437.73
ridge,Office Supplies,48,1503.12,39.22,-46103.03
random_forest,Office Supplies,48,1766.26,46.09,-38079.21
gradient_boosted_trees,Office Supplies,48,1779.65,46.44,-54230.04
ridge,Technology,48,2607.82,55.3,-59849.02
random_forest,Technology,48,2771.89,58.78,-58575.08
gradient_boosted_trees,Technology,48,3197.82,67.81,-47778.51


In [0]:
%python
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.sql import functions as F

raw = spark.table("workspace.default.superstore_features_month_region_category")
df = raw.select(
    "sales_month", "region", "category", "calendar_month", "observed_in_source",
    F.col("sales").cast("double").alias("label"),
    F.col("sales_lag_1").cast("double").alias("sales_lag_1"),
    F.col("sales_lag_12").cast("double").alias("sales_lag_12"),
    F.col("sales_trailing_3").cast("double").alias("sales_trailing_3"),
).where(
    F.col("sales_lag_1").isNotNull()
    & F.col("sales_lag_12").isNotNull()
    & F.col("sales_trailing_3").isNotNull()
)
train = df.where(F.year("sales_month").isin(2015, 2016))
test = df.where(F.year("sales_month") == 2017)
assert train.count() == 288 and test.count() == 144

indexers = [
    StringIndexer(inputCol=c, outputCol=c + "_idx", handleInvalid="keep")
    for c in ["region", "category", "calendar_month"]
]
encoder = OneHotEncoder(
    inputCols=[c + "_idx" for c in ["region", "category", "calendar_month"]],
    outputCols=[c + "_vec" for c in ["region", "category", "calendar_month"]],
    handleInvalid="keep",
)
assembler = VectorAssembler(
    inputCols=["region_vec", "category_vec", "calendar_month_vec",
               "sales_lag_1", "sales_lag_12", "sales_trailing_3"],
    outputCol="features",
)
ridge = LinearRegression(
    featuresCol="features", labelCol="label",
    regParam=1.0, elasticNetParam=0.0, maxIter=100,
)
fitted = Pipeline(stages=indexers + [encoder, assembler, ridge]).fit(train)
scored = fitted.transform(test).select(
    "sales_month", "region", "category", "observed_in_source", "label",
    F.greatest(F.lit(0.0), F.col("prediction")).alias("ridge_prediction"),
    F.when(F.col("category") == "Office Supplies", F.col("sales_trailing_3"))
     .otherwise(F.col("sales_lag_12")).alias("baseline_prediction"),
)
long = scored.select(
    "sales_month", "region", "category", "observed_in_source", "label",
    F.lit("ridge").alias("model"),
    F.col("ridge_prediction").alias("predicted"),
).unionByName(scored.select(
    "sales_month", "region", "category", "observed_in_source", "label",
    F.lit("best_2016_baseline").alias("model"),
    F.col("baseline_prediction").alias("predicted"),
))
metrics = long.groupBy("model", "category").agg(
    F.count("*").alias("observations"),
    F.round(F.avg(F.abs(F.col("label") - F.col("predicted"))), 2).alias("mae_usd"),
    F.round(
        100 * F.sum(F.abs(F.col("label") - F.col("predicted")))
        / F.sum("label"), 2
    ).alias("wape_pct"),
    F.round(F.sum(F.col("predicted") - F.col("label")), 2).alias("signed_bias_usd"),
)
display(metrics.orderBy("category", "wape_pct"))

worst = long.where(F.col("model") == "ridge").select(
    "sales_month", "region", "category", "observed_in_source", "label",
    "predicted",
    F.round(F.abs(F.col("label") - F.col("predicted")), 2).alias("abs_error_usd"),
).orderBy(F.desc("abs_error_usd")).limit(10)
display(worst)

model,category,observations,mae_usd,wape_pct,signed_bias_usd
ridge,Furniture,48,1853.93,41.32,-28541.99
best_2016_baseline,Furniture,48,1984.45,44.22,-16485.83
ridge,Office Supplies,48,2094.78,40.86,-81707.4
best_2016_baseline,Office Supplies,48,2730.47,53.26,-2014.55
ridge,Technology,48,3158.49,55.79,-73399.07
best_2016_baseline,Technology,48,4306.96,76.08,-45366.63


sales_month,region,category,observed_in_source,label,predicted,abs_error_usd
2017-10-01,East,Technology,true,22939.794,5285.729815517749,17654.06
2017-03-01,West,Technology,true,20351.684,5042.490001511659,15309.19
2017-11-01,East,Technology,true,22584.223,8247.095603606402,14337.13
2017-01-01,Central,Office Supplies,true,12704.288,956.8752870784383,11747.41
2017-08-01,West,Office Supplies,true,14860.29,3136.596837766685,11723.69
2017-11-01,South,Technology,true,15456.584,5276.713406433922,10179.87
2017-07-01,West,Technology,true,13870.854,4039.270622578204,9831.58
2017-11-01,West,Furniture,true,14154.679,7546.4758336354425,6608.2
2017-10-01,West,Office Supplies,true,10958.429,4465.298415208292,6493.13
2017-08-01,East,Office Supplies,true,9556.474,3428.347435635991,6128.13


In [0]:
%python
import base64
import gzip
import hashlib
import json
from decimal import Decimal
from pyspark.sql import functions as F

TABLE = "workspace.default.superstore_gold_month_region_category"
records = spark.table(TABLE).select(
    F.date_format("sales_month", "yyyy-MM").alias("month"),
    "region", "category", "order_lines", "units", "sales", "profit"
).orderBy("month", "region", "category").collect()

# Fixed-point integers: one unit equals USD 0.0001.
scale = Decimal("10000")
rows = [
    [r["month"], r["region"], r["category"],
     int(r["order_lines"]), int(r["units"]),
     int(Decimal(str(r["sales"])) * scale),
     int(Decimal(str(r["profit"])) * scale)]
    for r in records
]
assert len(rows) == 573
assert sum(r[3] for r in rows) == 9994

payload = {
    "schema": "superstore-month-region-category/v1",
    "scale": 10000,
    "columns": ["month", "region", "category", "order_lines",
                "units", "sales", "profit"],
    "rows": rows,
}
raw = json.dumps(payload, ensure_ascii=False, separators=(",", ":")).encode("utf-8")
print("ROWS", len(rows))
print("SALES", str(Decimal(sum(r[5] for r in rows)) / scale))
print("PROFIT", str(Decimal(sum(r[6] for r in rows)) / scale))
print("SHA256", hashlib.sha256(raw).hexdigest())
print("PAYLOAD_BASE64_GZIP")
print(base64.b64encode(gzip.compress(raw, mtime=0)).decode("ascii"))

ROWS 573
SALES 2297200.8603
PROFIT 286397.0217
SHA256 56e35ffaad2e6cd78a52f88d884579c0dead727ab809a82ed50f050c685b5576
PAYLOAD_BASE64_GZIP
H4sIAAAAAAACA41dy64lN3L8FaPX58J8ZSY5W8PeejEGvBgIhiD3zAjQdAv9sGEY/ndHZLXvYbGSdSyM1HeAjlssMpnPyKz//vD1l79+/NvPH/7w4ev33z9++frt85ePb3/7/OnbX9++fPzLr58/vf3y87ePf/n85b/+/j/yhwf++s+/ffzwh5zwz+PDL59/+/63T18//OFPHxyDv3Cg8MP/4fDj5y///vHLv/3266ePX/H/vn/69Rv//IrfxD9///L5z79++/ATsJ//k7/rTx9Kyu0t8Xn/8PHTty8//4af/un7FwC/f/n44dEeOT8kaZWeHiWNUuWnR4z65z//+ddfPv7dH7///vtvv/JxWR+tP7D+UhvQ3XpW26H/5eMvf/30+bfPf8Fb5Ed91Fz43sMkpTPmH3/++u28yPrI5ZHHSKmlx1vJmqSHmMsS9ZGxwlyG4Vm5ljFaCFxXl0vz5b3VnC4L/OPn734+0wr7o5ZHq9atYCdyrqmI9hh23Ud7CM5hiDTDC9Y6aq0ag+eF2qOORxnYd+wMtt+SrbB//bjupTwy/peltyp8v9Jy6ilEXReaH21guT2VjtfEqWNvxhNbbqSsYnsGdjM9tCUbtkNdHwqkPYoWFWxOHjYfYdnKWHmMh0DI+MRSujXTM2wnMgWH2VuDkOHRufUews4CY9zRqnhUG5raAgnkJVMyISIFUvYwqVZLjLksr2J5gg3E8mruZQObl4fNbw8r2e8Anme95TPqKiXFUS1Tmt9yN80lhISnhSO22qlQshiERUPoeuVKHb7CrrWP6Wl1J1QQf3lUraMMv6lJSm9b4PWcsfE4t9Kz37q3UhRXKe/w83Irb8JI2iuVp